In [2]:
# !pip install -q openpyxl tqdm

from openpyxl import load_workbook
from tqdm.auto import tqdm
import re
import html
import unicodedata

# =========================
# 1. File paths
# =========================

# Change filename if your uploaded file has (1), e.g.
EXPORT_FILE = "/content/franckenachlass_3.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

CORRECTED_LCNAF_FILE = "/content/LCNAF_updated.xlsx"

OUTPUT_FILE = "/content/franckenachlass_4.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

HEADER_ROW = 1

# Four columns where existing LC links should be replaced
CREATOR_LINKED_NEW = "creator_linked_new"
RECEIVER_LINKED_NEW = "receiver_linked_new"
SENDER_PLACE_LINKED_NEW = "sender_place_linked_new"
RECEIVER_PLACE_LINKED_NEW = "receiver_place_linked_new"

TARGET_REPLACE_COLUMNS = [
    CREATOR_LINKED_NEW,
    RECEIVER_LINKED_NEW,
    SENDER_PLACE_LINKED_NEW,
    RECEIVER_PLACE_LINKED_NEW
]

# If *_new columns do not exist, create them from original columns
FALLBACK_ORIGINAL_LINKED_COLS = {
    CREATOR_LINKED_NEW: "creator_linked",
    RECEIVER_LINKED_NEW: "receiver_linked",
    SENDER_PLACE_LINKED_NEW: "sender_place_linked",
    RECEIVER_PLACE_LINKED_NEW: "receiver_place_linked",
}

# Export file columns for insertion matching
RECEIVER_COL = "receiver"

# Corrected LCNAF file columns
PERSON_NAME_COL = "personStdName"
#PERSON_NAME_COL = "organizationDesc_original"

# This code supports both names because your files have used both versions
OLD_LOC_COL_CANDIDATES = ["LODLOC_inDB", "LODLOC"]

NEW_LOC_URI_COL = "LODLOC_uri"
NEW_LOC_LABEL_COL = "LODLOC_label"


# =========================
# 2. Helper functions
# =========================

def get_header_positions(ws):
    """
    Handles duplicate column names.
    Returns:
    {
        "creator_linked": [12, 50],
        "creator_linked_new": [80]
    }
    """
    positions = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(HEADER_ROW, col).value
        if value is not None:
            header = str(value).strip()
            positions.setdefault(header, []).append(col)
    return positions


def get_first_col(header_positions, col_name):
    if col_name not in header_positions:
        raise ValueError(f"Missing required column: {col_name}")
    return header_positions[col_name][0]


def get_first_available_col(header_positions, candidates):
    for col_name in candidates:
        if col_name in header_positions:
            return col_name, header_positions[col_name][0]
    raise ValueError(f"Missing all candidate columns: {candidates}")


def is_blank(value):
    return value is None or str(value).strip() == ""


def normalize_space(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_text_for_match(text):
    if text is None:
        return ""

    text = str(text).strip().strip("[]")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_person_name(name):
    if name is None:
        return ""

    name = str(name).strip()
    name = name.strip("[]")
    name = normalize_space(name)
    return name


def strip_life_dates_for_match(name):
    """
    Removes life dates only for matching.

    Examples:
    Archer, John, -1642 -> Archer, John
    Rost, Gottfried 1677-1753 -> Rost, Gottfried
    Smith, John, 1730-1800 -> Smith, John
    """
    if name is None:
        return ""

    name = str(name)

    # Remove parenthetical dates, e.g. (1650-1702)
    name = re.sub(
        r"\s*\([^)]*(?:\d{3,4}|b\.|d\.|fl\.|ca\.|c\.)[^)]*\)",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Remove comma date fragments, e.g. , -1642 or , 1730-1800
    name = re.sub(
        r"\s*,\s*-?\d{3,4}\s*(?:[-–]\s*\d{0,4})?",
        "",
        name
    )

    # Remove plain trailing dates
    name = re.sub(r"\b\d{3,4}\s*[-–]\s*\d{0,4}\b", "", name)
    name = re.sub(r"\b\d{3,4}\b", "", name)

    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s+,", ",", name).strip()

    return name


def person_std_to_full_name(person_std):
    """
    Converts:
    Archer, John
    to:
    John Archer
    """
    person_std = clean_person_name(person_std)
    person_std = strip_life_dates_for_match(person_std)

    if "," not in person_std:
        return person_std

    last, first = person_std.split(",", 1)

    last = normalize_space(last)
    first = normalize_space(first)

    if first and last:
        return f"{first} {last}"

    return person_std


def make_author_std_name(fname, mname, lname):
    """
    Converts authorN_fname + optional authorN_mname + authorN_lname to:
    Last, First Middle

    Examples:
    John + William + Archer -> Archer, John William
    John + blank + Archer   -> Archer, John
    """
    fname = normalize_space(fname)
    mname = normalize_space(mname)
    lname = normalize_space(lname)

    first_parts = []

    if fname:
        first_parts.append(fname)

    if mname:
        first_parts.append(mname)

    first_middle = normalize_space(" ".join(first_parts))

    if lname and first_middle:
        return f"{lname}, {first_middle}"

    if lname:
        return lname

    if first_middle:
        return first_middle

    return ""


def extract_loc_id(value):
    """
    Handles:
    https://id.loc.gov/authorities/names/n84121476
    https://id.loc.gov/authorities/names/n84121476.html
    http://id.loc.gov/authorities/names/n84121476
    n84121476
    """
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    # If the value itself is an LCNAF id
    if re.fullmatch(r"[a-z]{1,4}\d+[a-z0-9]*", text, flags=re.IGNORECASE):
        return text.replace(".html", "").strip()

    m = re.search(
        r"id\.loc\.gov/authorities/names/([A-Za-z0-9]+)(?:\.html)?",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1)

    return None


def normalize_loc_uri(uri):
    """
    LCNAF official URI should use http, not https.
    """
    loc_id = extract_loc_id(uri)

    if loc_id:
        return f"http://id.loc.gov/authorities/names/{loc_id}"

    if uri is None:
        return ""

    return str(uri).strip().replace("https://id.loc.gov", "http://id.loc.gov").replace(".html", "").rstrip("/")


def build_lc_anchor(uri, label):
    uri = html.escape(normalize_loc_uri(uri), quote=True)
    label = html.escape(str(label).strip(), quote=False)
    return f'<p><a href="{uri}">{label}</a> [LC]</p>'


def append_lc_anchor_if_missing(cell_value, anchor_html, new_loc_id):
    """
    Appends LC anchor only if the same LCNAF id is not already in the cell.
    """
    if is_blank(cell_value):
        return anchor_html, True

    cell_text = str(cell_value)

    existing_ids = set()

    for m in LCNAF_ANCHOR_RE.finditer(cell_text):
        existing_id = extract_loc_id(m.group("href"))
        if existing_id:
            existing_ids.add(existing_id)

    if new_loc_id in existing_ids:
        return cell_text, False

    return cell_text + anchor_html, True


def get_or_create_target_col(ws, header_positions, new_col_name):
    """
    If *_new column exists, use it.
    If it does not exist, create it by copying the original linked column.
    """
    if new_col_name in header_positions:
        return header_positions[new_col_name][0]

    fallback_col_name = FALLBACK_ORIGINAL_LINKED_COLS[new_col_name]

    if fallback_col_name not in header_positions:
        raise ValueError(
            f"Missing both '{new_col_name}' and fallback '{fallback_col_name}'"
        )

    fallback_idx = header_positions[fallback_col_name][0]

    new_idx = ws.max_column + 1
    ws.cell(HEADER_ROW, new_idx).value = new_col_name

    for row in range(HEADER_ROW + 1, ws.max_row + 1):
        ws.cell(row, new_idx).value = ws.cell(row, fallback_idx).value

    header_positions[new_col_name] = [new_idx]

    print(f"Created '{new_col_name}' from '{fallback_col_name}'")

    return new_idx


def find_author_pairs(header_positions):
    """
    Finds all authorN_fname + optional authorN_mname + authorN_lname groups.
    Middle name column is optional.
    """
    author_nums = set()

    for header in header_positions:
        m = re.fullmatch(r"author(\d+)_(?:fname|mname|lname)", header)
        if m:
            author_nums.add(int(m.group(1)))

    pairs = []

    for n in sorted(author_nums):
        fname_col = f"author{n}_fname"
        mname_col = f"author{n}_mname"
        lname_col = f"author{n}_lname"

        if fname_col in header_positions or lname_col in header_positions:
            pairs.append({
                "author_num": n,
                "fname_col": fname_col if fname_col in header_positions else None,
                "mname_col": mname_col if mname_col in header_positions else None,
                "lname_col": lname_col if lname_col in header_positions else None,
                "fname_idx": header_positions[fname_col][0] if fname_col in header_positions else None,
                "mname_idx": header_positions[mname_col][0] if mname_col in header_positions else None,
                "lname_idx": header_positions[lname_col][0] if lname_col in header_positions else None,
            })

    return pairs


def add_to_multimap(mapping, key, record):
    if not key:
        return
    mapping.setdefault(key, []).append(record)


# =========================
# 3. Regex for LCNAF anchors
# =========================

LCNAF_ANCHOR_RE = re.compile(
    r'(?P<start><a\b[^>]*\bhref=["\']'
    r'(?P<href>https?://id\.loc\.gov/authorities/names/(?P<locid>[A-Za-z0-9]+)(?:\.html)?[^"\']*)'
    r'["\'][^>]*>)'
    r'(?P<label>.*?)'
    r'(?P<end></a>)',
    re.IGNORECASE | re.DOTALL
)


# =========================
# 4. Load corrected LCNAF file
# =========================

lc_wb = load_workbook(CORRECTED_LCNAF_FILE, data_only=True)

# Auto-detect sheet containing required columns
lc_ws = None
lc_headers = None

for ws in lc_wb.worksheets:
    headers = get_header_positions(ws)

    has_old_col = any(c in headers for c in OLD_LOC_COL_CANDIDATES)

    required = [
        PERSON_NAME_COL,
        NEW_LOC_URI_COL,
        NEW_LOC_LABEL_COL
    ]

    if has_old_col and all(col in headers for col in required):
        lc_ws = ws
        lc_headers = headers
        break

if lc_ws is None:
    raise ValueError("Could not find corrected LCNAF sheet with required columns.")

print("Using corrected LCNAF sheet:", lc_ws.title)

person_name_idx = get_first_col(lc_headers, PERSON_NAME_COL)
old_loc_indexes = []

for candidate in OLD_LOC_COL_CANDIDATES:
    if candidate in lc_headers:
        old_loc_indexes.extend(lc_headers[candidate])

print("Using old LCNAF columns:", old_loc_indexes)
new_loc_uri_idx = get_first_col(lc_headers, NEW_LOC_URI_COL)
new_loc_label_idx = get_first_col(lc_headers, NEW_LOC_LABEL_COL)

# Case 1: old LODLOC exists -> replace existing LCNAF anchors
replace_by_old_loc_id = {}

# NEW:
# Also allow matching existing LCNAF anchors that already use the corrected/new LCNAF ID.
# This fixes rows that appear in lcnaf_existing_missing even though the ID is in the corrected file.
replace_by_new_loc_id = {}

# Case 2: name-based insertion map
# Important: EVERY corrected record is added here, even if old LODLOC exists.
insert_by_creator_key = {}
insert_by_receiver_key = {}

lc_records = []
duplicate_old_ids = []

for row in range(HEADER_ROW + 1, lc_ws.max_row + 1):
    person_name_raw = lc_ws.cell(row, person_name_idx).value
    old_loc_raw = None

    for idx in old_loc_indexes:
        value = lc_ws.cell(row, idx).value
        if not is_blank(value):
            old_loc_raw = value
            break

    new_uri_raw = lc_ws.cell(row, new_loc_uri_idx).value
    new_label_raw = lc_ws.cell(row, new_loc_label_idx).value

    if is_blank(new_uri_raw) or is_blank(new_label_raw):
        continue

    # Use personStdName if available.
    # If personStdName is blank, fall back to LODLOC_label.
    if not is_blank(person_name_raw):
        person_std = clean_person_name(person_name_raw)
    else:
        person_std = strip_life_dates_for_match(str(new_label_raw).strip())

    person_full = person_std_to_full_name(person_std)

    old_loc_id = extract_loc_id(old_loc_raw)
    new_loc_id = extract_loc_id(new_uri_raw)

    if not new_loc_id:
        continue

    record = {
        "corrected_lcnaf_row": row,
        "personStdName": person_std,
        "person_full_name": person_full,
        "old_lodloc": old_loc_raw,
        "old_loc_id": old_loc_id,
        "new_loc_uri": normalize_loc_uri(new_uri_raw),
        "new_loc_id": new_loc_id,
        "new_loc_label": str(new_label_raw).strip(),
    }

    lc_records.append(record)

    # Replacement map using old LCNAF ID
    if old_loc_id:
        if old_loc_id in replace_by_old_loc_id:
            duplicate_old_ids.append(old_loc_id)

        replace_by_old_loc_id[old_loc_id] = record

    # NEW:
    # Replacement map using new/corrected LCNAF ID.
    # This catches cases where the export already has the corrected LCNAF URI,
    # but the label still needs to be updated.
    if new_loc_id:
        replace_by_new_loc_id[new_loc_id] = record

    # Insertion map:
    # Add EVERY record to name-based insertion maps.
    # This fixes the case where old LODLOC exists in corrected_lcnaf,
    # but that old LC link is absent from the export/main file.
    creator_name_clean = strip_life_dates_for_match(person_std)
    receiver_full_clean = strip_life_dates_for_match(person_full)

    # Creator matching uses Last, First
    add_to_multimap(
        insert_by_creator_key,
        normalize_text_for_match(creator_name_clean),
        record
    )

    # Receiver matching uses First Last
    add_to_multimap(
        insert_by_receiver_key,
        normalize_text_for_match(receiver_full_clean),
        record
    )

    # Also support receiver written as Last, First
    add_to_multimap(
        insert_by_receiver_key,
        normalize_text_for_match(creator_name_clean),
        record
    )

print("Loaded LCNAF records:", len(lc_records))
print("Records for replacing old LCNAF:", len(replace_by_old_loc_id))
print("Creator insertion keys:", len(insert_by_creator_key))
print("Receiver insertion keys:", len(insert_by_receiver_key))
print("Duplicate old LODLOC ids:", len(set(duplicate_old_ids)))


# =========================
# 5. Load export file
# =========================

export_wb = load_workbook(EXPORT_FILE)

# Auto-detect export sheet
export_ws = None
export_headers = None

for ws in export_wb.worksheets:
    headers = get_header_positions(ws)

    has_linked_cols = (
        CREATOR_LINKED_NEW in headers
        or FALLBACK_ORIGINAL_LINKED_COLS[CREATOR_LINKED_NEW] in headers
    )

    has_receiver = RECEIVER_COL in headers

    has_author = any(
        re.fullmatch(r"author\d+_(?:fname|mname|lname)", h)
        for h in headers
    )

    if has_linked_cols and has_receiver and has_author:
        export_ws = ws
        export_headers = headers
        break

if export_ws is None:
    raise ValueError(
        "Could not find export sheet with linked, receiver, and author columns."
    )

print("Using export sheet:", export_ws.title)

creator_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    CREATOR_LINKED_NEW
)

receiver_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    RECEIVER_LINKED_NEW
)

sender_place_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    SENDER_PLACE_LINKED_NEW
)

receiver_place_linked_new_idx = get_or_create_target_col(
    export_ws,
    export_headers,
    RECEIVER_PLACE_LINKED_NEW
)

receiver_idx = get_first_col(export_headers, RECEIVER_COL)

author_pairs = find_author_pairs(export_headers)

if not author_pairs:
    raise ValueError("No authorN_fname/mname/lname groups found.")

print("Found author groups:")

for pair in author_pairs:
    n = pair["author_num"]

    parts = []

    if pair.get("fname_idx") is not None:
        parts.append(f"author{n}_fname")

    if pair.get("mname_idx") is not None:
        parts.append(f"author{n}_mname")

    if pair.get("lname_idx") is not None:
        parts.append(f"author{n}_lname")

    print("  " + " + ".join(parts))

target_replace_columns = [
    ("creator_linked_new", creator_linked_new_idx),
    ("receiver_linked_new", receiver_linked_new_idx),
    ("sender_place_linked_new", sender_place_linked_new_idx),
    ("receiver_place_linked_new", receiver_place_linked_new_idx),
]

print("Using linked columns:")

for name, idx in target_replace_columns:
    print(f"  {name}: Excel column {idx}")


# =========================
# 6. Replace existing LCNAF anchors in four columns
# =========================

replace_log = []
existing_lc_not_in_correction_log = []
used_record_rows = set()
total_replaced = 0


def replace_lcnaf_in_cell(cell_value, row_num, col_name):
    if cell_value is None or not isinstance(cell_value, str):
        return cell_value, []

    if "id.loc.gov/authorities/names/" not in cell_value.lower():
        return cell_value, []

    changes = []

    def repl(match):
        old_href = match.group("href")
        old_label = match.group("label")
        old_loc_id = extract_loc_id(old_href)

        rec = (
            replace_by_old_loc_id.get(old_loc_id)
            or replace_by_new_loc_id.get(old_loc_id)
        )

        if rec is None:
            existing_lc_not_in_correction_log.append([
                row_num,
                col_name,
                old_loc_id,
                old_href,
                old_label
            ])
            return match.group(0)

        new_href = rec["new_loc_uri"]
        new_label = rec["new_loc_label"]

        # =========================================================
        # ONLY CHANGE:
        # Do not count/log a replacement when URI and label are
        # already exactly the corrected values.
        # =========================================================
        normalized_old_href = normalize_loc_uri(old_href)
        normalized_new_href = normalize_loc_uri(new_href)

        normalized_old_label = html.unescape(
            str(old_label)
        ).strip()

        normalized_new_label = html.unescape(
            str(new_label)
        ).strip()

        if (
            normalized_old_href == normalized_new_href
            and normalized_old_label == normalized_new_label
        ):
            return match.group(0)
        # =========================================================
        # END OF ONLY CHANGE
        # =========================================================

        escaped_new_href = html.escape(new_href, quote=True)
        escaped_new_label = html.escape(new_label, quote=False)

        changes.append([
            row_num,
            col_name,
            rec["corrected_lcnaf_row"],
            rec["personStdName"],
            old_loc_id,
            old_href,
            old_label,
            new_href,
            new_label
        ])

        used_record_rows.add(rec["corrected_lcnaf_row"])

        return f'<a href="{escaped_new_href}">{escaped_new_label}</a>'

    new_value = LCNAF_ANCHOR_RE.sub(repl, cell_value)

    return new_value, changes


for col_name, col_idx in target_replace_columns:
    for row in tqdm(
        range(HEADER_ROW + 1, export_ws.max_row + 1),
        desc=f"Replacing LCNAF in {col_name}"
    ):
        cell = export_ws.cell(row, col_idx)
        old_value = cell.value

        new_value, changes = replace_lcnaf_in_cell(
            old_value,
            row,
            col_name
        )

        if changes:
            cell.value = new_value
            total_replaced += len(changes)
            replace_log.extend(changes)

print("Total existing LCNAF anchors replaced:", total_replaced)


# =========================
# 7. Insert missing LCNAF into creator_linked_new and receiver_linked_new only
# =========================

insert_log = []
ambiguous_insert_log = []
used_insert_record_rows = set()
total_inserted = 0


def insert_records_into_cell(
    row,
    target_col_idx,
    target_col_name,
    records,
    match_source
):
    global total_inserted

    if not records:
        return

    # De-duplicate by new LCNAF id
    unique = {}

    for rec in records:
        unique[rec["new_loc_id"]] = rec

    records = list(unique.values())

    if len(records) > 1:
        ambiguous_insert_log.append([
            row,
            target_col_name,
            match_source,
            "; ".join(
                f'{rec["corrected_lcnaf_row"]}: '
                f'{rec["personStdName"]} -> {rec["new_loc_uri"]}'
                for rec in records
            )
        ])
        return

    rec = records[0]

    anchor = build_lc_anchor(
        rec["new_loc_uri"],
        rec["new_loc_label"]
    )

    cell = export_ws.cell(row, target_col_idx)
    old_cell_value = cell.value

    new_cell_value, did_insert = append_lc_anchor_if_missing(
        old_cell_value,
        anchor,
        rec["new_loc_id"]
    )

    if did_insert:
        cell.value = new_cell_value
        total_inserted += 1

        used_record_rows.add(rec["corrected_lcnaf_row"])
        used_insert_record_rows.add(rec["corrected_lcnaf_row"])

        insert_log.append([
            row,
            target_col_name,
            match_source,
            rec["corrected_lcnaf_row"],
            rec["personStdName"],
            rec["person_full_name"],
            rec["old_lodloc"],
            rec["old_loc_id"],
            rec["new_loc_id"],
            rec["new_loc_uri"],
            rec["new_loc_label"],
            old_cell_value,
            new_cell_value
        ])


for row in tqdm(
    range(HEADER_ROW + 1, export_ws.max_row + 1),
    desc="Inserting missing LCNAF"
):

    # -------- creator insertion: author1, author2, author3, ... --------

    for pair in author_pairs:
        fname = (
            export_ws.cell(row, pair["fname_idx"]).value
            if pair.get("fname_idx") is not None
            else None
        )

        mname = (
            export_ws.cell(row, pair["mname_idx"]).value
            if pair.get("mname_idx") is not None
            else None
        )

        lname = (
            export_ws.cell(row, pair["lname_idx"]).value
            if pair.get("lname_idx") is not None
            else None
        )

        author_std = make_author_std_name(
            fname,
            mname,
            lname
        )

        if is_blank(author_std):
            continue

        author_key = normalize_text_for_match(
            strip_life_dates_for_match(author_std)
        )

        records = insert_by_creator_key.get(
            author_key,
            []
        )

        if records:
            insert_records_into_cell(
                row=row,
                target_col_idx=creator_linked_new_idx,
                target_col_name="creator_linked_new",
                records=records,
                match_source=(
                    f"author{pair['author_num']}: "
                    f"{author_std}"
                )
            )

    # -------- receiver insertion --------

    receiver_value = export_ws.cell(
        row,
        receiver_idx
    ).value

    if not is_blank(receiver_value):
        receiver_text = normalize_space(
            receiver_value
        )

        possible_receiver_parts = [
            receiver_text
        ]

        # Split only on strong separators.
        # Do not split on comma because names may be Last, First.
        possible_receiver_parts.extend(
            [
                part.strip()
                for part in re.split(
                    r"[;|\n/]+",
                    receiver_text
                )
                if part.strip()
            ]
        )

        seen_receiver_parts = set()

        for receiver_part in possible_receiver_parts:
            key = normalize_text_for_match(
                strip_life_dates_for_match(
                    receiver_part
                )
            )

            if not key or key in seen_receiver_parts:
                continue

            seen_receiver_parts.add(key)

            records = insert_by_receiver_key.get(
                key,
                []
            )

            if records:
                insert_records_into_cell(
                    row=row,
                    target_col_idx=receiver_linked_new_idx,
                    target_col_name="receiver_linked_new",
                    records=records,
                    match_source=(
                        f"receiver: {receiver_part}"
                    )
                )

print(
    "Total new LCNAF anchors inserted:",
    total_inserted
)

print(
    "Ambiguous insert cases skipped:",
    len(ambiguous_insert_log)
)


# =========================
# 7.5 Convert all existing LCNAF hrefs to official http form
# =========================

from urllib.parse import unquote

ANY_ANCHOR_RE = re.compile(
    r'(?P<start><a\b[^>]*\bhref=["\'](?P<href>[^"\']+)["\'][^>]*>)'
    r'(?P<label>.*?)'
    r'(?P<end></a>)',
    re.IGNORECASE | re.DOTALL
)


def decode_possible_url(text):
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)

    for _ in range(3):
        decoded = unquote(text)

        if decoded == text:
            break

        text = decoded

    return text


def extract_loc_id_from_any_text(value):
    """
    Extract LCNAF ID from direct http/https URL, .html URL, encoded URL,
    Google-search URL text, or bare LCNAF ID.
    """
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(
        r"[a-z]{1,4}\d+[a-z0-9]*",
        text,
        flags=re.IGNORECASE
    ):
        return text.replace(".html", "").strip()

    text = decode_possible_url(text)

    m = re.search(
        r"id\.loc\.gov/authorities/names/([A-Za-z0-9]+)(?:\.html)?",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1)

    return None


def clean_lcnaf_hrefs_to_http(
    cell_value,
    row_num,
    col_name
):
    """
    Converts any LCNAF anchor href to:
    http://id.loc.gov/authorities/names/<ID>

    Does not change label text.
    Does not touch VIAF, Wikidata, GeoNames, etc.
    """
    if cell_value is None or not isinstance(
        cell_value,
        str
    ):
        return cell_value, []

    if (
        "id.loc.gov" not in cell_value.lower()
        and
        "id.loc.gov" not in
        decode_possible_url(cell_value).lower()
    ):
        return cell_value, []

    changes = []

    def repl(match):
        href = match.group("href")
        label = match.group("label")

        loc_id = extract_loc_id_from_any_text(
            href
        )

        # Fallback: sometimes ID/URL may be visible in label text
        if loc_id is None:
            loc_id = extract_loc_id_from_any_text(
                label
            )

        if loc_id is None:
            return match.group(0)

        old_anchor = match.group(0)

        new_href = (
            f"http://id.loc.gov/"
            f"authorities/names/{loc_id}"
        )

        # Replace only the href value,
        # keep label and [LC] text unchanged
        new_anchor = re.sub(
            r'href=["\'][^"\']+["\']',
            f'href="{html.escape(new_href, quote=True)}"',
            old_anchor,
            count=1,
            flags=re.IGNORECASE
        )

        if new_anchor != old_anchor:
            changes.append([
                row_num,
                col_name,
                loc_id,
                href,
                new_href,
                label
            ])

        return new_anchor

    new_value = ANY_ANCHOR_RE.sub(
        repl,
        cell_value
    )

    return new_value, changes


lc_http_cleanup_log = []

for col_name, col_idx in target_replace_columns:
    for row in tqdm(
        range(
            HEADER_ROW + 1,
            export_ws.max_row + 1
        ),
        desc=(
            f"Converting LCNAF https to http "
            f"in {col_name}"
        )
    ):
        cell = export_ws.cell(
            row,
            col_idx
        )

        old_value = cell.value

        new_value, changes = clean_lcnaf_hrefs_to_http(
            old_value,
            row,
            col_name
        )

        if changes:
            cell.value = new_value
            lc_http_cleanup_log.extend(
                changes
            )

print(
    "LCNAF hrefs converted to http:",
    len(lc_http_cleanup_log)
)


# =========================
# 8. Add log sheets
# =========================

log_sheet_names = [
    "lcnaf_replace_log",
    "lcnaf_insert_log",
    "lcnaf_ambiguous_insert_log",
    "lcnaf_existing_missing",
    "lcnaf_duplicate_old_ids",
    "lcnaf_not_applied"
]

for sheet_name in log_sheet_names:
    if sheet_name in export_wb.sheetnames:
        del export_wb[sheet_name]


# Replacement log
ws_replace = export_wb.create_sheet(
    "lcnaf_replace_log"
)

ws_replace.append([
    "export_row",
    "column",
    "corrected_lcnaf_row",
    "personStdName",
    "old_lcnaf_id",
    "old_lcnaf_href",
    "old_lcnaf_label",
    "new_lcnaf_href",
    "new_lcnaf_label"
])

for item in replace_log:
    ws_replace.append(item)


# Insert log
ws_insert = export_wb.create_sheet(
    "lcnaf_insert_log"
)

ws_insert.append([
    "export_row",
    "target_column",
    "match_source",
    "corrected_lcnaf_row",
    "personStdName",
    "person_full_name",
    "old_lodloc",
    "old_lcnaf_id",
    "new_lcnaf_id",
    "new_lcnaf_href",
    "new_lcnaf_label",
    "old_cell_value",
    "new_cell_value"
])

for item in insert_log:
    ws_insert.append(item)


# Ambiguous insertion log
ws_ambiguous = export_wb.create_sheet(
    "lcnaf_ambiguous_insert_log"
)

ws_ambiguous.append([
    "export_row",
    "target_column",
    "match_source",
    "possible_lcnaf_records"
])

for item in ambiguous_insert_log:
    ws_ambiguous.append(item)


# Existing LCNAF anchors not in correction replacement map
ws_existing = export_wb.create_sheet(
    "lcnaf_existing_missing"
)

ws_existing.append([
    "export_row",
    "column",
    "existing_lcnaf_id",
    "existing_lcnaf_href",
    "existing_lcnaf_label"
])

seen_existing = set()

for item in existing_lc_not_in_correction_log:
    key = tuple(item)

    if key in seen_existing:
        continue

    seen_existing.add(key)

    ws_existing.append(item)


# Duplicate old ids
ws_dupes = export_wb.create_sheet(
    "lcnaf_duplicate_old_ids"
)

ws_dupes.append([
    "duplicate_old_lcnaf_id"
])

for loc_id in sorted(
    set(duplicate_old_ids)
):
    ws_dupes.append([loc_id])


# Records that were neither replaced nor inserted
ws_not_applied = export_wb.create_sheet(
    "lcnaf_not_applied"
)

ws_not_applied.append([
    "corrected_lcnaf_row",
    "personStdName",
    "person_full_name",
    "old_lodloc",
    "old_lcnaf_id",
    "new_lcnaf_id",
    "new_lcnaf_href",
    "new_lcnaf_label",
    "reason"
])

for rec in lc_records:
    if rec["corrected_lcnaf_row"] not in used_record_rows:
        ws_not_applied.append([
            rec["corrected_lcnaf_row"],
            rec["personStdName"],
            rec["person_full_name"],
            rec["old_lodloc"],
            rec["old_loc_id"],
            rec["new_loc_id"],
            rec["new_loc_uri"],
            rec["new_loc_label"],
            (
                "Not replaced and not inserted. "
                "No safe creator/receiver name match, "
                "skipped as ambiguous, or not present in export."
            )
        ])


# Make log sheets readable
for ws_log in [
    ws_replace,
    ws_insert,
    ws_ambiguous,
    ws_existing,
    ws_dupes,
    ws_not_applied
]:
    for col in range(
        1,
        ws_log.max_column + 1
    ):
        ws_log.column_dimensions[
            ws_log.cell(
                1,
                col
            ).column_letter
        ].width = 34


# =========================
# 9. Save output
# =========================

# Make sure all sheets are visible
for ws in export_wb.worksheets:
    ws.sheet_state = "visible"

# Make the main export sheet active
export_wb.active = export_wb.sheetnames.index(
    export_ws.title
)

# Reset workbook view settings safely
if export_wb.views:
    export_wb.views[0].activeTab = (
        export_wb.sheetnames.index(
            export_ws.title
        )
    )

    export_wb.views[0].firstSheet = 0

export_wb.save(
    OUTPUT_FILE
)

print("Done.")
print("Saved file:", OUTPUT_FILE)

Using corrected LCNAF sheet: Sheet1
Using old LCNAF columns: [1]
Loaded LCNAF records: 238
Records for replacing old LCNAF: 29
Creator insertion keys: 238
Receiver insertion keys: 469
Duplicate old LODLOC ids: 0
Using export sheet: 40427936
Found author groups:
  author1_fname + author1_mname + author1_lname
  author2_fname + author2_mname + author2_lname
  author3_fname + author3_mname + author3_lname
  author4_fname + author4_mname + author4_lname
  author5_fname + author5_mname + author5_lname
Using linked columns:
  creator_linked_new: Excel column 39
  receiver_linked_new: Excel column 40
  sender_place_linked_new: Excel column 41
  receiver_place_linked_new: Excel column 42


Replacing LCNAF in creator_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing LCNAF in receiver_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing LCNAF in sender_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing LCNAF in receiver_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

...

Converting LCNAF https to http in creator_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Converting LCNAF https to http in receiver_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Converting LCNAF https to http in sender_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Converting LCNAF https to http in receiver_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

LCNAF hrefs converted to http: 0
Done.
Saved file: /content/franckenachlass_4.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
